# FAQ Matching Bot — Train & export Keras model

**Steps:** 1) load libraries → 2) dataset → 3) text normalizer (stemming + lemmatization) → 4) train → 5) download Keras model

*How it works:* text is normalized (stop-words removed, words reduced to their lemma and stem), then TF-IDF turns it into vectors. The Keras model stores every FAQ vector in one layer; multiplying a
question vector by that layer gives its **cosine similarity** to every FAQ.

## 1. Load libraries

In [ ]:
import json, zipfile
from datetime import datetime, timezone

import joblib, keras, pandas as pd, sklearn
from keras import layers
from sklearn.feature_extraction.text import TfidfVectorizer

## 2. Dataset = written dataset + your own CSV (merged)

* The **written dataset** below is always used.
* Your **own CSV** (optional) is uploaded in the next cell and **added** to it — it does not replace it.
  Required columns: `question`, `answer` (optional: `category`, `id`, and `keywords`). If an answer contains commas, wrap it in
  double quotes (`"like, this"`) — Excel/Sheets "Save as CSV" does this automatically.
* **`keywords` (optional, recommended):** synonyms and word forms for each FAQ, separated by spaces
  (e.g. `coupon coupons promo promotional discount offer`). TF-IDF only matches words, so this is how the bot learns that
  "coupon" means the same as "promo code". It raised accuracy on our test questions from about 46% to 87%.

In [ ]:
%%writefile written_faqs.csv
id,category,question,answer,keywords
1,Orders,How do I track my order?,"Open 'My Orders' in your account and click 'Track package'. You will also find a tracking link in your shipping confirmation email, which is sent within 24 hours of dispatch.",tracking track tracked parcel package shipment status whereabouts location trace follow
2,Orders,How can I cancel my order?,"You can cancel for free from 'My Orders' as long as the order has not shipped yet. Once it has shipped, please refuse delivery or start a return after it arrives.",cancel cancelling cancellation cancelled stop abort undo remove purchase
3,Orders,Can I change my shipping address after placing an order?,"Yes, but only before the order is dispatched. Go to 'My Orders' > 'Edit address' or contact support right away. We cannot redirect a parcel that is already in transit.",address wrong incorrect typo update edit modify destination redirect deliver where order going
4,Orders,I received a damaged or incorrect item. What should I do?,We are sorry about that! Send us a photo of the item and your order number via the 'Report a problem' button within 7 days of delivery. We will ship a replacement or issue a full refund.,broken faulty defective wrong missing damaged damage arrived received replacement complaint product
5,Orders,Why is my order delayed?,"Delays can be caused by high demand, bad weather or customs checks. Check the live tracking page for updates. If your order is more than 3 business days late, contact support and we will investigate.",late delay delays delayed overdue slow arrived yet waiting stuck parcel not arrived
6,Shipping,How long does delivery take?,Standard delivery takes 3-5 business days. Express delivery takes 1-2 business days. Remote areas may take up to 2 extra days.,days duration time arrive arrival eta fast speed deliver delivery times how fast quickly
7,Shipping,How much does shipping cost?,"Standard shipping costs $4.99, and it is free on orders over $50. Express shipping costs $12.99.",price prices charge charges fee fees free postage cost costs how much expensive
8,Shipping,Do you ship internationally?,"Yes, we ship to over 40 countries. International delivery takes 7-14 business days, and import duties or taxes may be charged by your local customs office.",overseas abroad worldwide foreign countries country outside europe asia america global international export
9,Shipping,Do you offer same-day or express delivery?,Express delivery (1-2 business days) is available at checkout for most addresses. Same-day delivery is currently available in selected cities for orders placed before 12:00 noon.,next day fast faster urgent quick rush tomorrow priority overnight expedited
10,Returns,What is your return policy?,"You can return most items within 30 days of delivery for a full refund, provided they are unused, unwashed and in their original packaging with tags attached.",refund refunds returns window send back policy guarantee money back days
11,Returns,How do I return an item?,"Go to 'My Orders', choose the item and click 'Return'. Print the prepaid label, pack the item securely and drop it at any partner courier location.",send back return returning returns steps label courier procedure how to
12,Returns,When will I get my refund?,Refunds are issued to your original payment method within 5-7 business days after we receive and inspect your return. Your bank may need additional time to post it.,money back reimbursement reimbursed credit refunded refunds returned
13,Returns,Can I exchange an item for a different size or color?,"Yes. Start a return and select 'Exchange' to choose another size or color. If the new item costs more, you will only pay the difference.",swap switch replace replacement different bigger smaller size colour color change
14,Payments,Which payment methods do you accept?,"We accept Visa, Mastercard, American Express, PayPal, Apple Pay, Google Pay and gift cards.",pay payments paying cards card credit debit paypal apple google visa mastercard accept options methods
15,Payments,Is it safe to use my credit card on your website?,Yes. All payments are encrypted with SSL/TLS and processed by PCI-DSS compliant providers. We never store your full card number on our servers.,secure security safe safety encrypted protected fraud privacy ssl trust card details
16,Payments,My payment failed. What should I do?,"Please check your card details, billing address and available balance. If the problem continues, try another payment method or contact your bank, as they may have blocked the transaction.",declined decline failed error rejected refused transaction not working issue problem
17,Payments,How do I use a discount code or promo code?,"Enter your code in the 'Promo code' box on the checkout page and click 'Apply'. Only one code can be used per order, and some items may be excluded.",use using apply applying enter entering redeem redeeming checkout box field type coupon coupons promo promos promotional codes discount discounts
18,Payments,Can I pay in installments?,"Yes. Choose 'Pay in 4' at checkout to split your purchase into four interest-free payments, available for orders between $50 and $1,000.",installment installments instalment instalments split later monthly financing plan parts buy now pay later
19,Account,How do I create an account?,"Click 'Sign up' at the top right of the page, enter your name, email and a password, then confirm your email address using the link we send you.",sign up signup register registration join open new profile membership
20,Account,I forgot my password. How can I reset it?,Click 'Log in' > 'Forgot password' and enter your email. We will send you a reset link that stays valid for 30 minutes.,forgotten reset recover recovery login log in sign in locked out access cannot remember passcode credentials
21,Account,How do I update my email address or phone number?,Sign in and go to 'Account settings' > 'Profile'. Edit your email or phone number and click 'Save'. You will need to verify a new email address.,change edit modify email phone number contact details profile mobile
22,Account,How do I delete my account?,Go to 'Account settings' > 'Privacy' > 'Delete account'. Your personal data will be erased within 30 days. Order records required by law may be retained.,close remove deactivate erase terminate account permanently data
23,Account,How do I unsubscribe from marketing emails?,"Click the 'Unsubscribe' link at the bottom of any promotional email, or turn off 'Marketing emails' under 'Account settings' > 'Notifications'.",newsletter newsletters promotional emails spam stop sending opt out marketing mail subscription
24,Products,How do I know if a product is in stock?,The product page shows 'In stock' or 'Out of stock'. For sold-out items click 'Notify me' and we will email you as soon as it is back.,stock availability available sold out restock back in stock notify inventory
25,Products,Do your products come with a warranty?,Electronics come with a 12-month manufacturer warranty. Other products are covered against defects for 90 days. Keep your order confirmation as proof of purchase.,guarantee guaranteed warranty defect repair covered coverage electronics
26,Products,How do I find the right size?,"Each product page has a 'Size guide' link with measurements in cm and inches. If you are between sizes, we recommend sizing up.",sizing sizes fit chart measurements measure clothing clothes shoes small medium large
27,Support,How can I contact customer support?,"You can chat with us live from the bottom right of the page, email support@shopease.example, or call 1-800-555-0199.",contact support help customer service chat email phone call agent reach talk
28,Support,What are your customer support hours?,"Our team is available Monday to Friday, 9:00 to 18:00 (local time), and Saturday 10:00 to 16:00. Email is monitored 24/7 with replies within one business day.",hours open opening times schedule when call working customer service
29,Support,Do you have a physical store I can visit?,"We are an online-only retailer, but we run pop-up stores in selected cities. Check the 'Events' page for upcoming locations.",shop shops stores retail locations visit in person branch showroom outlet walk in
30,Gift cards,Do you sell gift cards and how do I redeem them?,"Digital gift cards from $10 to $500 are available on the 'Gift cards' page. To redeem one, enter the code in the 'Gift card' box at checkout.",gift card cards voucher vouchers certificate redeem balance present
31,Promotions,Where can I find coupon codes and current offers?,"All current offers and coupon codes are listed in the 'Offers' banner on the home page. Subscribe to our newsletter to get exclusive codes by email, then enter the code at checkout.",give get find where available any current offers offer deals deal sale sales coupon coupons code codes voucher promo promotions promotional free today list home page banner newsletter latest discount discounts running ongoing active
32,Promotions,Do you offer a discount on my first order?,"Yes. New customers get 10% off their first order with the code WELCOME10, valid for 30 days after signing up.",new customer first time welcome sign up signup discount code coupon offer percent off beginner
33,Promotions,Do you have a sale or clearance section?,Yes. Open 'Sale' in the main menu to see reduced items. Clearance stock is marked 'Final sale' and cannot be returned.,sale sales clearance reduced reductions cheap cheaper bargain bargains markdown markdowns deals discounted outlet price prices products items cheaper lower
34,Promotions,Do you have a loyalty or rewards program?,Yes. Join ShopEase Rewards for free in 'Account settings' and earn 1 point per $1 spent. 100 points give you a $5 voucher.,rewards reward points point loyalty member membership club earn collect perks vip

In [ ]:
USE_OWN_DATA = True     # True: a file picker opens - choose your CSV(s). Click Cancel / set False to use only the written dataset.

own_files = []
if USE_OWN_DATA:
    from google.colab import files
    own_files = list(files.upload().keys())
print("own files:", own_files or "none")

In [ ]:
from pathlib import Path


def read_faqs(path):
    bad = []
    df = pd.read_csv(path, encoding="utf-8-sig", engine="python",
                     on_bad_lines=lambda row: bad.append(row) or None).fillna("")   # skip (and report) malformed lines
    df.columns = [str(c).strip().lower() for c in df.columns]
    if not {"question", "answer"} <= set(df.columns):
        raise ValueError(f"'{path}' doesn't look like an FAQ file: it needs 'question' and 'answer' columns but has "
                         f"{list(df.columns)}. Upload a CSV of questions with their answers (or set USE_OWN_DATA = False).")
    if "category" not in df:
        df["category"] = "General"
    if "keywords" not in df:
        df["keywords"] = ""
    if bad:
        print(f"WARNING {path}: skipped {len(bad)} malformed line(s) - wrap answers that contain commas in double quotes. "
              f"First one: {bad[0][:3]}")
    df["source"] = Path(path).name
    return df[["category", "question", "answer", "keywords", "source"]]


# own data first, so on a duplicate question your version wins over the written one
frames = [read_faqs(f) for f in own_files if f.lower().endswith(".csv")] + [read_faqs("written_faqs.csv")]
faqs = pd.concat(frames, ignore_index=True)
faqs["question"] = faqs["question"].astype(str).str.strip()
faqs = faqs[(faqs["question"] != "") & (faqs["answer"].astype(str).str.strip() != "")]
faqs = faqs.drop_duplicates(subset="question", keep="first").reset_index(drop=True)
faqs.insert(0, "id", range(1, len(faqs) + 1))               # fresh ids, so the two files can't collide
faqs.to_csv("faqs.csv", index=False)                          # merged dataset (also goes in the download)

print(f"{len(faqs)} FAQs in total:")
print(faqs["source"].value_counts().to_string())
faqs.head()

## 3. Text normalizer: stemming + lemmatization

TF-IDF only matches identical words, so "refunding", "refunded" and "refunds" would count as three different words.
The normalizer reduces them to one: it removes stop-words, **lemmatizes** irregular forms (`bought → buy`,
`children → child`) and **stems** the rest with the Porter algorithm (`shipping, shipped, ships → ship`).
The same code runs in the backend and in the browser version, so training and answering always agree.
(The code is hidden: click *Show code* to read it.)

In [ ]:
#@title Normalizer code (click "Show code" to read)
NORMALIZER_SRC = r'''"""Text normalisation for the FAQ matcher: stop-words -> lemmatisation -> stemming.

    "I was refunded for the returned parcels"  ->  "refund return parcel"

Why: TF-IDF only matches identical tokens. Normalising maps different forms of a word to one token, so a question
that uses a form the FAQ never contained ("refunding", "cancelled", "coupons") still matches.

Pipeline (the same code runs in the backend, the Colab notebook and - ported line by line - in the browser):
  1. lower-case, strip accents, drop apostrophes ("can't" -> "cant")
  2. split into word tokens of 2+ characters
  3. drop English stop-words (before and after lemmatising)
  4. lemmatise: irregular forms via a dictionary ("bought" -> "buy", "children" -> "child", "paid" -> "pay")
  5. stem: Porter algorithm ("shipping"/"shipped"/"ships" -> "ship", "codes" -> "code")

Dependencies: scikit-learn (only for its stop-word list). Bump PREPROCESS if you change anything below so that
models trained with an older normaliser are not mixed with this one.
"""
import re
import unicodedata

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

PREPROCESS = "stem_lemma_v2"
# scikit-learn's list misses some very common function and chat words ("does" appears in many FAQs and would make
# unrelated questions match them), so add those.
EXTRA_STOP_WORDS = frozenset({
    "does", "did", "doing", "doesn", "didn", "don", "isn", "wasn", "aren", "weren", "hasn", "haven", "hadn",
    "wouldn", "couldn", "shouldn", "please", "pls", "plz", "hi", "hello", "hey", "thanks", "thank", "thx",
    "want", "wanted", "wants", "need", "needed", "needs", "like", "tell", "let", "ok", "okay", "just",
    "really", "also",
})
STOP_WORDS = frozenset(ENGLISH_STOP_WORDS) | EXTRA_STOP_WORDS

# ---------------------------------------------------------------------------------------------------------------
# Lemmatisation: irregular forms that no suffix rule can handle (dictionary-based, like WordNet's exception lists)
# ---------------------------------------------------------------------------------------------------------------
LEMMAS = {
    # irregular verbs
    "arose": "arise", "arisen": "arise", "ate": "eat", "eaten": "eat", "began": "begin", "begun": "begin",
    "bitten": "bite", "blew": "blow", "blown": "blow", "bought": "buy", "brought": "bring",
    "built": "build", "broke": "break", "broken": "break", "came": "come", "caught": "catch", "chose": "choose",
    "chosen": "choose", "dealt": "deal", "drew": "draw", "drawn": "draw", "drove": "drive", "driven": "drive",
    "fell": "fall", "fallen": "fall", "felt": "feel", "flew": "fly", "flown": "fly", "forgot": "forget",
    "forgotten": "forget", "found": "find", "gave": "give", "given": "give", "goes": "go", "went": "go",
    "gone": "go", "got": "get", "gotten": "get", "grew": "grow", "grown": "grow", "held": "hold", "hid": "hide",
    "hidden": "hide", "kept": "keep", "knew": "know", "known": "know", "laid": "lay", "lent": "lend", "lost": "lose", "made": "make", "met": "meet", "paid": "pay", "ran": "run",
    "rang": "ring", "rung": "ring", "risen": "rise", "saw": "see", "seen": "see", "sent": "send",
    "sold": "sell", "spent": "spend", "spoke": "speak", "spoken": "speak", "stole": "steal", "stolen": "steal",
    "swore": "swear", "sworn": "swear", "taught": "teach", "told": "tell", "took": "take", "taken": "take",
    "thought": "think", "threw": "throw", "thrown": "throw", "understood": "understand", "woke": "wake",
    "woken": "wake", "won": "win", "wore": "wear", "worn": "wear", "wrote": "write", "written": "write",
    "bigger": "big", "biggest": "big",
    # irregular nouns
    "children": "child", "men": "man", "women": "woman", "people": "person", "feet": "foot", "teeth": "tooth",
    "mice": "mouse", "geese": "goose", "knives": "knife", "wives": "wife", "shelves": "shelf", "halves": "half",
    "wolves": "wolf", "analyses": "analysis", "indices": "index", "criteria": "criterion",
    "selves": "self", "loaves": "loaf", "thieves": "thief",
    # spelling variants (UK / US) that Porter would stem differently
    "colour": "color", "colours": "color", "favourite": "favorite", "cheque": "check", "grey": "gray",
    "cancelled": "cancel", "cancelling": "cancel", "labelled": "label", "travelled": "travel", "jewellery": "jewelry",
    "instalment": "installment", "instalments": "installment", "centre": "center", "litre": "liter",
    "harbour": "harbor", "neighbour": "neighbor", "neighbours": "neighbor", "programme": "program",
    "personalise": "personalize", "personalised": "personalize", "personalising": "personalize",
    "customise": "customize", "customised": "customize", "customising": "customize",
    "organise": "organize", "organised": "organize", "organising": "organize",
    "recognise": "recognize", "recognised": "recognize", "authorise": "authorize", "authorised": "authorize",
    "apologise": "apologize", "apologised": "apologize", "catalogue": "catalog", "licence": "license",
}

# ---------------------------------------------------------------------------------------------------------------
# Porter stemmer (regex formulation of M. Porter's 1980 algorithm). The JavaScript port in standalone/tfidf.js
# mirrors this code exactly; tests/test_standalone.py checks both give the same output.
# ---------------------------------------------------------------------------------------------------------------
_STEP2 = {
    "ational": "ate", "tional": "tion", "enci": "ence", "anci": "ance", "izer": "ize", "bli": "ble", "alli": "al",
    "entli": "ent", "eli": "e", "ousli": "ous", "ization": "ize", "ation": "ate", "ator": "ate", "alism": "al",
    "iveness": "ive", "fulness": "ful", "ousness": "ous", "aliti": "al", "iviti": "ive", "biliti": "ble",
    "logi": "log",
}
_STEP3 = {"icate": "ic", "ative": "", "alize": "al", "iciti": "ic", "ical": "ic", "ful": "", "ness": ""}

_c = "[^aeiou]"
_v = "[aeiouy]"
_C = _c + "[^aeiouy]*"
_V = _v + "[aeiou]*"
_MGR0 = re.compile("^(" + _C + ")?" + _V + _C)
_MEQ1 = re.compile("^(" + _C + ")?" + _V + _C + "(" + _V + ")?$")
_MGR1 = re.compile("^(" + _C + ")?" + _V + _C + _V + _C)
_S_V = re.compile("^(" + _C + ")?" + _v)
_CVC = re.compile("^" + _C + _v + "[^aeiouwxy]$")
_DOUBLE = re.compile("([^aeiouylsz])\\1$")
_STEP2_RE = re.compile("^(.+?)(" + "|".join(_STEP2) + ")$")
_STEP3_RE = re.compile("^(.+?)(" + "|".join(_STEP3) + ")$")
_STEP4_RE = re.compile("^(.+?)(al|ance|ence|er|ic|able|ible|ant|ement|ment|ent|ou|ism|ate|iti|ous|ive|ize)$")
_STEP4_ION_RE = re.compile("^(.+?)(s|t)(ion)$")


def stem(w: str) -> str:
    """Porter-stem one lower-case word. Anything that is not purely a-z is returned unchanged."""
    if len(w) < 3 or not re.fullmatch("[a-z]+", w):
        return w
    first = w[0]
    if first == "y":
        w = "Y" + w[1:]          # an initial y is a consonant

    # step 1a: plurals
    m = re.search("^(.+?)(ss|i)es$", w)
    if m:
        w = m.group(1) + m.group(2)
    else:
        m = re.search("^(.+?)([^s])s$", w)
        if m:
            w = m.group(1) + m.group(2)

    # step 1b: -eed / -ed / -ing
    m = re.search("^(.+?)eed$", w)
    if m:
        if _MGR0.search(m.group(1)):
            w = w[:-1]
    else:
        m = re.search("^(.+?)(ed|ing)$", w)
        if m and _S_V.search(m.group(1)):
            w = m.group(1)
            if re.search("(at|bl|iz)$", w):
                w += "e"
            elif _DOUBLE.search(w):
                w = w[:-1]
            elif _CVC.search(w):
                w += "e"

    # step 1c: y -> i
    m = re.search("^(.+?)y$", w)
    if m and _S_V.search(m.group(1)):
        w = m.group(1) + "i"

    # step 2
    m = _STEP2_RE.search(w)
    if m and _MGR0.search(m.group(1)):
        w = m.group(1) + _STEP2[m.group(2)]

    # step 3
    m = _STEP3_RE.search(w)
    if m and _MGR0.search(m.group(1)):
        w = m.group(1) + _STEP3[m.group(2)]

    # step 4
    m = _STEP4_RE.search(w)
    if m:
        if _MGR1.search(m.group(1)):
            w = m.group(1)
    else:
        m = _STEP4_ION_RE.search(w)
        if m and _MGR1.search(m.group(1) + m.group(2)):
            w = m.group(1) + m.group(2)

    # step 5
    m = re.search("^(.+?)e$", w)
    if m:
        s = m.group(1)
        if _MGR1.search(s) or (_MEQ1.search(s) and not _CVC.search(s)):
            w = s
    if re.search("ll$", w) and _MGR1.search(w):
        w = w[:-1]

    return "y" + w[1:] if first == "y" else w


# ---------------------------------------------------------------------------------------------------------------
def _strip_accents(s: str) -> str:
    normalized = unicodedata.normalize("NFKD", s)
    return "".join(c for c in normalized if not unicodedata.combining(c))


def lemmatize(token: str) -> str:
    return LEMMAS.get(token, token)


def tokens(text: str) -> list:
    """Normalised tokens of a text: stop-words removed, lemmatised, stemmed."""
    text = _strip_accents(str(text).lower()).replace("'", "").replace("’", "")
    out = []
    for tok in re.findall(r"\w\w+", text):
        lemma = lemmatize(tok)
        if tok in STOP_WORDS or lemma in STOP_WORDS:     # e.g. "got" -> "get" is a stop-word too
            continue
        out.append(stem(lemma))
    return out


def normalize(text: str) -> str:
    """Normalised text, tokens separated by spaces (feed this to a whitespace-tokenising TfidfVectorizer)."""
    return " ".join(tokens(text))
'''
with open("normalizer.py", "w") as f:
    f.write(NORMALIZER_SRC)

from normalizer import normalize, PREPROCESS

for text in ["I was refunded for the returned parcels", "Cancelling orders & coupons", "my children bought it"]:
    print(f"{text!r:45} -> {normalize(text)!r}")

## 4. Train

In [ ]:
# Each FAQ = question (x2) + answer + keywords, normalized (stop-words out, lemmatized, stemmed).
# TF-IDF vectors are L2-normalised, so dot product = cosine similarity.
texts = ((faqs["question"] + " ") * 2 + faqs["answer"] + " " + faqs["keywords"]).map(normalize)
CONFIG = dict(lowercase=False, strip_accents=None, stop_words=None, token_pattern=r"\S+", ngram_range=(1, 1), sublinear_tf=True)

vectorizer = TfidfVectorizer(**CONFIG)
faq_matrix = vectorizer.fit_transform(texts)              # (n_faqs, vocab)
n_faqs, vocab = faq_matrix.shape

# Keras model: TF-IDF query vector -> cosine similarity with every FAQ
inputs = keras.Input(shape=(vocab,), name="tfidf_query")
outputs = layers.Dense(n_faqs, use_bias=False, trainable=False, name="cosine_similarity")(inputs)
model = keras.Model(inputs, outputs, name="faq_matcher")
model.get_layer("cosine_similarity").set_weights([faq_matrix.toarray().T.astype("float32")])
model.summary()


def ask(question, top_k=3):
    scores = model.predict(vectorizer.transform([normalize(question)]).toarray(), verbose=0)[0]
    for i in scores.argsort()[::-1][:top_k]:
        print(f"  {scores[i]:.2f}  {faqs.loc[i, 'question']}")


# the second and third questions use word forms that appear nowhere in the FAQs ("refunding", "coupons")
for q in ["I forgot my login password", "refunding returned items", "any coupons?"]:
    print("Q:", q); ask(q)

## 5. Save & download

Downloads **`faq_model_keras.zip`** containing `faq_model.keras`, `tfidf_vectorizer.joblib`, `faq_meta.json` and the merged `faqs.csv`.
Unzip it into the project's `model/` folder and restart the backend.

In [ ]:
model.save("faq_model.keras")
joblib.dump(vectorizer, "tfidf_vectorizer.joblib")
meta = {
    "faqs": faqs[["id", "category", "question", "answer"]].to_dict("records"),
    "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "sklearn_version": sklearn.__version__,
    "keras_version": keras.__version__,
    "preprocess": PREPROCESS,          # tells the backend which text normalizer this model was trained with
}
with open("faq_meta.json", "w") as f:
    json.dump(meta, f, default=int)

with zipfile.ZipFile("faq_model_keras.zip", "w") as z:
    for name in ["faq_model.keras", "tfidf_vectorizer.joblib", "faq_meta.json", "faqs.csv"]:
        z.write(name)

try:
    from google.colab import files
    files.download("faq_model_keras.zip")
except ImportError:
    print("Saved faq_model_keras.zip in the current folder (not running in Colab).")